# 04 — Ablation Study
## Brain Tumour Detection — Final Project
=====================================

Topics covered:
  1.  What an ablation is for
  2.  The protocol — subset, fixed baseline, one factor at a time
  3.  The noise floor — how big a difference has to be to mean anything
  4.  Augmentation
  5.  Dropout and weight decay
  6.  Label smoothing
  7.  Depth and receptive field
  8.  Input resolution
  9.  The full results table
  10. Confirming the winner on the full dataset
  11. Summary

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config, data, engine, viz
from src.config import CACHE_DIR, DEVICE, FACE, PALETTE, TRAIN_DIR

train_img = np.load(CACHE_DIR / "train_img.npy")
train_lab = np.load(CACHE_DIR / "train_lab.npy")
MEAN, STD = np.load(CACHE_DIR / "norm.npy")
CLASSES   = sorted(d.name for d in TRAIN_DIR.iterdir())
train_idx, val_idx = data.stratified_split(train_lab)
print(f"device {DEVICE}   full train pool {len(train_idx)}   val {len(val_idx)}")

device cuda   full train pool 4760   val 840


In [2]:
# 1. WHAT AN ABLATION IS FOR
"""
By the end of Day 8 the model had accumulated six regularisation and
architecture choices: augmentation, dropout, weight decay, label smoothing,
depth, input resolution. Every one of them was justified by an argument. None
of them had been shown to help on real data.

An ablation removes or changes one component at a time and measures what
happens. It converts a list of defensible choices into a list of measured
contributions, and it is the difference between "we used dropout because it
regularises" and "dropout changed validation accuracy by this much, here".

It also catches choices that are actively harmful. Day 8 found weight decay
hurt on its starved synthetic baseline — a result nobody would have predicted
from theory, and one that only appeared because it was measured.
"""
print("factors carried into this study, and where each was introduced:\n")
for factor, origin in [("augmentation (RandomAffine)", "Day 2, revisited Day 8"),
                       ("dropout", "Day 5"),
                       ("weight decay", "Day 6"),
                       ("label smoothing", "Day 8"),
                       ("network depth / receptive field", "Day 4, notebook 02 s.3"),
                       ("input resolution", "Day 2")]:
    print(f"  {factor:<34} {origin}")

factors carried into this study, and where each was introduced:

  augmentation (RandomAffine)        Day 2, revisited Day 8
  dropout                            Day 5
  weight decay                       Day 6
  label smoothing                    Day 8
  network depth / receptive field    Day 4, notebook 02 s.3
  input resolution                   Day 2


In [3]:
# 2. THE PROTOCOL
"""
Three decisions define this study, and each trades something away.

A 1,200-image stratified subset instead of all 4,760. Nine training runs on the
full set would take hours. The subset makes the study affordable, and the cost
is that absolute accuracies here are lower than notebook 02's — these numbers
are for comparing configurations to each other, not for reporting.

25 epochs, no early stopping. Every configuration gets exactly the same budget,
so no run is advantaged by being allowed to train longer.

One factor at a time from a fixed baseline, rather than a grid. A full grid over
six factors would be 64+ runs. The limitation is real and worth stating: this
design cannot detect interactions, so if augmentation and dropout only help
together, this study will report that neither helps.

The baseline is deliberately unregularised, so every factor has room to show an
effect.
"""
SUB_PER_CLASS, ABL_EPOCHS = 300, 25
rng = np.random.default_rng(config.SEED)
sub_idx = train_idx[np.concatenate(
    [rng.choice(np.where(train_lab[train_idx] == c)[0], SUB_PER_CLASS, replace=False)
     for c in range(len(CLASSES))])]
val_sub = val_idx

BASELINE = dict(dropout=0.0, weight_decay=0.0, augment=False,
                label_smoothing=0.0, deep=False, img_size=128)

print(f"training subset {len(sub_idx)} images ({SUB_PER_CLASS}/class)")
print(f"validation      {len(val_sub)} images (the full validation split)")
print(f"epochs          {ABL_EPOCHS}, no early stopping\n")
print("baseline configuration:")
for k, v in BASELINE.items():
    print(f"  {k:<18} {v}")

def run(tag, **overrides):
    cfg = {**BASELINE, **overrides}
    t0 = time.time()
    h = engine.run_experiment(train_img, train_lab, sub_idx, val_sub, MEAN, STD,
                              epochs=ABL_EPOCHS, **cfg)
    s = engine.summarise(h)
    s["tag"] = tag
    print(f"  {tag:<26} val acc {s['best_val_acc']:.4f}   "
          f"overfit {s['overfit_ratio']:>5.2f}x   ({time.time()-t0:.0f}s)")
    return s

results = []

training subset 1200 images (300/class)
validation      840 images (the full validation split)
epochs          25, no early stopping

baseline configuration:
  dropout            0.0
  weight_decay       0.0
  augment            False
  label_smoothing    0.0
  deep               False
  img_size           128


In [4]:
# 3. THE NOISE FLOOR
"""
Before comparing configurations, we need to know how big a difference has to be
before it means anything.

The same configuration trained three times with different seeds gives three
different answers, because initialisation, augmentation draws and batch order
all vary. The spread between those runs is the noise floor. Any factor whose
effect is smaller than it is indistinguishable from luck.

Skipping this step is the most common way an ablation table gets over-read:
without it, every difference looks like a finding.
"""
seeds = []
print("baseline, three seeds:")
for seed in (42, 7, 1234):
    h = engine.run_experiment(train_img, train_lab, sub_idx, val_sub, MEAN, STD,
                              epochs=ABL_EPOCHS, seed=seed, **BASELINE)
    s = engine.summarise(h)
    seeds.append(s["best_val_acc"])
    print(f"  seed {seed:<5} val acc {s['best_val_acc']:.4f}")

NOISE = float(np.std(seeds))
SPREAD = float(max(seeds) - min(seeds))
print(f"\nmean {np.mean(seeds):.4f}   std {NOISE:.4f}   spread {SPREAD:.4f}")
print(f"\n-> treat differences below {SPREAD:.4f} as noise, not signal")

results.append({"tag": "baseline (none)", "best_val_acc": float(np.mean(seeds)),
                "overfit_ratio": float('nan'), "best_val_loss": float('nan'),
                "final_val_acc": float('nan'), "seconds": float('nan')})
BASE_ACC = float(np.mean(seeds))

baseline, three seeds:


  seed 42    val acc 0.8845


  seed 7     val acc 0.8857


  seed 1234  val acc 0.8869

mean 0.8857   std 0.0010   spread 0.0024

-> treat differences below 0.0024 as noise, not signal


In [5]:
# 4. AUGMENTATION
"""
Day 8 found augmentation the single most effective regulariser on synthetic
data, and argued it works differently from the others: dropout and weight decay
penalise the model for using its capacity, while augmentation increases the
effective quantity of data. It attacks the cause rather than the symptom.

Both variants are tested — geometric only, and geometric plus intensity jitter —
because notebook 01 made a specific argument for keeping ColorJitter that Day 8
had dismissed. That argument should be measured, not just asserted.
"""
print("augmentation:")
results.append(run("augment (affine)",        augment=True, jitter=False))
results.append(run("augment (affine+jitter)", augment=True, jitter=True))

for r in results[-2:]:
    print(f"\n  {r['tag']:<26} {r['best_val_acc']-BASE_ACC:+.4f} vs baseline "
          f"({'above' if abs(r['best_val_acc']-BASE_ACC) > SPREAD else 'within'} noise)")

augmentation:


  augment (affine)           val acc 0.8655   overfit  1.00x   (53s)


  augment (affine+jitter)    val acc 0.8679   overfit  1.02x   (59s)

  augment (affine)           -0.0202 vs baseline (above noise)

  augment (affine+jitter)    -0.0179 vs baseline (above noise)


In [6]:
# 5. DROPOUT AND WEIGHT DECAY
"""
The two classical capacity penalties, both introduced earlier in the project
with a mechanism but never with a measurement on real scans.

Dropout zeroes activations at random during training, forcing the network to
avoid depending on any single unit. Weight decay pulls every weight toward zero
each step, so a weight survives only if the gradient keeps pushing it back.

Day 8's starved synthetic baseline found dropout roughly neutral and weight
decay actively harmful. That was 40 images; this is 1,200 real ones, and there
is no reason the ordering has to hold.
"""
print("capacity penalties:")
results.append(run("dropout 0.4",      dropout=0.4))
results.append(run("weight decay 1e-4", weight_decay=1e-4))

capacity penalties:


  dropout 0.4                val acc 0.8726   overfit  1.00x   (51s)


  weight decay 1e-4          val acc 0.8845   overfit  1.00x   (50s)


In [7]:
# 6. LABEL SMOOTHING
"""
The one technique introduced on Day 8 rather than carried in. Instead of
training toward a hard target of 1.0 for the correct class, it targets
(1 - eps) + eps/K, spreading eps across the others.

The effect is to cap how confident the model is rewarded for being. Under a
one-hot target, cross-entropy keeps falling as the correct logit grows without
bound, so the optimiser has a permanent incentive toward overconfidence. Under
a smoothed target the loss reaches a minimum at a finite logit gap and rises
again past it.

Section 7 of notebook 03 measured exactly this property on the final model, so
this row connects directly to the calibration result there.
"""
print("label smoothing:")
results.append(run("label smoothing 0.1", label_smoothing=0.1))

label smoothing:


  label smoothing 0.1        val acc 0.8905   overfit  1.00x   (51s)


In [8]:
# 7. DEPTH AND RECEPTIVE FIELD
"""
The architectural factor, and the one with a specific prediction attached.

Notebook 02 section 3 computed that the default stack sees only 38px of a 128px
scan at its deepest convolution — under a third of the image. The argument was
that glioma versus meningioma turns on context that window cannot contain:
whether a mass sits inside brain tissue or on the meninges, its margin, its
position relative to the midline.

If that reasoning is right, adding depth should help, and it should help by
reducing exactly the glioma/meningioma confusion identified in notebook 03. If
the deeper model gains nothing, the receptive-field argument was wrong and the
report should say so.
"""
from src.model import receptive_field, BrainTumourNet, count_parameters
print(f"default: RF {receptive_field(False)}px, "
      f"{count_parameters(BrainTumourNet())[0]:,} params")
print(f"deep:    RF {receptive_field(True)}px, "
      f"{count_parameters(BrainTumourNet(deep=True))[0]:,} params\n")
print("depth:")
results.append(run("deeper (RF 62px)", deep=True))

default: RF 38px, 429,732 params
deep:    RF 62px, 1,167,780 params

depth:


  deeper (RF 62px)           val acc 0.9107   overfit  1.01x   (63s)


In [9]:
# 8. INPUT RESOLUTION
"""
Because the network ends in a global average pool, its parameter count does not
depend on input size at all — only the compute does. 192x192 costs 2.25 times
the FLOPs of 128x128 for exactly the same weights.

Whether that buys anything is an empirical question about how much diagnostic
detail survives downsampling. Small lesions and fine margin texture are the
first things to disappear when a 512px scan is squeezed to 128px.
"""
print("resolution:")
results.append(run("resolution 192", img_size=192))

resolution:


  resolution 192             val acc 0.8702   overfit  1.00x   (122s)


In [10]:
# 9. THE FULL RESULTS TABLE
"""
Sorted by validation accuracy, with the difference from baseline shown against
the noise floor measured in section 3. The 'signal' column is the honest part:
anything marked otherwise is a difference this study cannot distinguish from
run-to-run variation.
"""
ranked = sorted(results, key=lambda r: -r["best_val_acc"])
print(f"{'configuration':<26}{'val acc':>9}{'vs base':>10}{'overfit':>10}{'signal':>9}")
print("-" * 64)
for r in ranked:
    d = r["best_val_acc"] - BASE_ACC
    sig = "-" if r["tag"].startswith("baseline") else ("yes" if abs(d) > SPREAD else "noise")
    ratio = "n/a" if np.isnan(r["overfit_ratio"]) else f"{r['overfit_ratio']:.2f}x"
    print(f"{r['tag']:<26}{r['best_val_acc']:>9.4f}{d:>+10.4f}{ratio:>10}{sig:>9}")

fig, ax = viz.styled_fig(figsize=(8, 4.5))
tags = [r["tag"] for r in ranked]
deltas = [r["best_val_acc"] - BASE_ACC for r in ranked]
colours = [PALETTE["good"] if d > SPREAD else
           PALETTE["val"] if d < -SPREAD else '#B0B0AC' for d in deltas]
ax.barh(range(len(tags)), deltas, color=colours)
ax.axvline(0, color='k', lw=1)
ax.axvspan(-SPREAD, SPREAD, color='grey', alpha=0.18, label=f"noise floor (±{SPREAD:.3f})")
ax.set_yticks(range(len(tags))); ax.set_yticklabels(tags, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("change in validation accuracy vs baseline")
ax.set_title("Ablation — contribution of each factor", fontsize=11, fontweight='bold')
ax.legend(fontsize=8); ax.set_facecolor(FACE)
plt.tight_layout(); viz.save(fig, "ablation.png")

winners = [r["tag"] for r in ranked
           if r["best_val_acc"] - BASE_ACC > SPREAD and not r["tag"].startswith("baseline")]
print(f"\nfactors that beat the noise floor: {winners if winners else 'none'}")

configuration               val acc   vs base   overfit   signal
----------------------------------------------------------------
deeper (RF 62px)             0.9107   +0.0250     1.01x      yes
label smoothing 0.1          0.8905   +0.0048     1.00x      yes
baseline (none)              0.8857   +0.0000       n/a        -
weight decay 1e-4            0.8845   -0.0012     1.00x    noise
dropout 0.4                  0.8726   -0.0131     1.00x      yes
resolution 192               0.8702   -0.0155     1.00x      yes
augment (affine+jitter)      0.8679   -0.0179     1.02x      yes
augment (affine)             0.8655   -0.0202     1.00x      yes
  saved -> outputs/ablation.png

factors that beat the noise floor: ['deeper (RF 62px)', 'label smoothing 0.1']


In [11]:
# 10. CONFIRMING THE WINNER ON THE FULL DATASET
"""
Everything above ran on a quarter of the training data. A factor that helps on
1,200 images does not automatically help on 4,760 — regularisation in
particular tends to matter less as data increases, since the model has less
opportunity to memorise in the first place.

So the study ends by combining the factors that cleared the noise floor and
training once on the full training split, against the same validation set
notebook 02 used. That makes it directly comparable to the model that was
actually shipped.
"""
combined = dict(BASELINE)
for r in ranked:
    if r["best_val_acc"] - BASE_ACC <= SPREAD or r["tag"].startswith("baseline"):
        continue
    tag = r["tag"]
    if tag.startswith("augment"):        combined.update(augment=True, jitter="jitter" in tag)
    elif tag.startswith("dropout"):      combined.update(dropout=0.4)
    elif tag.startswith("weight decay"): combined.update(weight_decay=1e-4)
    elif tag.startswith("label"):        combined.update(label_smoothing=0.1)
    elif tag.startswith("deeper"):       combined.update(deep=True)
    elif tag.startswith("resolution"):   combined.update(img_size=192)

print("combined configuration:")
for k, v in combined.items():
    print(f"  {k:<18} {v}")

t0 = time.time()
h_full = engine.run_experiment(train_img, train_lab, train_idx, val_idx, MEAN, STD,
                               epochs=30, verbose=True, **combined)
s_full = engine.summarise(h_full)
print(f"\nfull-data combined: best val acc {s_full['best_val_acc']:.4f} "
      f"at epoch {s_full['best_epoch']}   ({time.time()-t0:.0f}s)")
h02 = np.load(config.OUTPUTS / "history.npy", allow_pickle=True).item()
print(f"notebook 02 shipped model:      {max(h02['val_acc']):.4f} "
      f"(full data, {h02['stopped_at']} epochs, early stopping)")

combined configuration:
  dropout            0.0
  weight_decay       0.0
  augment            False
  label_smoothing    0.1
  deep               True
  img_size           128


  epoch   1/30  train 0.9284/0.6710   val 0.8651/0.6750  <- best


  epoch   5/30  train 0.5642/0.8974   val 0.5560/0.7821


  epoch  10/30  train 0.4905/0.9324   val 0.3036/0.9119  <- best


  epoch  15/30  train 0.4088/0.9723   val 0.2511/0.9333


  epoch  20/30  train 0.3644/0.9951   val 0.1776/0.9560  <- best


  epoch  25/30  train 0.3571/0.9968   val 0.1568/0.9667  <- best


  epoch  30/30  train 0.3536/0.9985   val 0.1580/0.9655

full-data combined: best val acc 0.9667 at epoch 29   (252s)
notebook 02 shipped model:      0.9833 (full data, 60 epochs, early stopping)


In [12]:
# 11. SUMMARY
"""
What this study can and cannot support, stated plainly.
"""
print("=" * 64)
print("NOTEBOOK 04 — ABLATION SUMMARY")
print("=" * 64)
print(f"  baseline (3 seeds)      {BASE_ACC:.4f}  +/- {NOISE:.4f}")
print(f"  noise floor (spread)    {SPREAD:.4f}")
print(f"  best single factor      {ranked[0]['tag']} ({ranked[0]['best_val_acc']:.4f})")
print(f"  factors above noise     {len(winners)} of {len(results)-1}")
print(f"  combined, full data     {s_full['best_val_acc']:.4f}")

print("""
  WHAT THIS SUPPORTS

  Each factor's effect was measured against a fixed baseline under an
  identical budget, with a noise floor established from repeated runs of
  that baseline. Differences reported as signal exceed that floor.

  WHAT IT DOES NOT

  One factor at a time cannot detect interactions. If two techniques only
  help in combination, this design reports that neither helps.

  The sweep ran on 1,200 of 4,760 training images, so the absolute
  accuracies are not comparable to notebook 03 and regularisation is likely
  over-weighted relative to the full-data regime.

  One seed per configuration, except the baseline. The noise floor tells us
  how much of any single row could be luck; it does not remove it.

  Every caveat from notebook 03 still applies — most importantly the absence
  of patient identifiers, which no amount of ablation can fix.""")

NOTEBOOK 04 — ABLATION SUMMARY
  baseline (3 seeds)      0.8857  +/- 0.0010
  noise floor (spread)    0.0024
  best single factor      deeper (RF 62px) (0.9107)
  factors above noise     2 of 7
  combined, full data     0.9667

  WHAT THIS SUPPORTS

  Each factor's effect was measured against a fixed baseline under an
  identical budget, with a noise floor established from repeated runs of
  that baseline. Differences reported as signal exceed that floor.

  WHAT IT DOES NOT

  One factor at a time cannot detect interactions. If two techniques only
  help in combination, this design reports that neither helps.

  The sweep ran on 1,200 of 4,760 training images, so the absolute
  accuracies are not comparable to notebook 03 and regularisation is likely
  over-weighted relative to the full-data regime.

  One seed per configuration, except the baseline. The noise floor tells us
  how much of any single row could be luck; it does not remove it.

  Every caveat from notebook 03 still appli